# KvForge + STAR-KV + LoRA-RKR

Kaggle'da STAR-KV adaptive low-rank + LoRA detail restoration testi.
CPU mode (P100 CC 6.0 incompatible).


In [ ]:
import sys, math, time, json, warnings, os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
device = 'cpu'
print(f'Device: {device}')


In [ ]:
class RandomSVD:
    @staticmethod
    def compute(X, k, n_oversamples=10, n_iter=2):
        m, n = X.shape[-2], X.shape[-1]
        p = min(k + n_oversamples, n)
        Q = torch.randn(n, p, device=X.device, dtype=X.dtype)
        Y = X @ Q
        for _ in range(n_iter):
            Y = X @ (X.mT @ Y)
            Y = torch.linalg.qr(Y).Q
        B = Y.mT @ X
        Ub, Sb, Vhb = torch.linalg.svd(B, full_matrices=False)
        return Y @ Ub[..., :k], Sb[..., :k], Vhb[..., :k, :]


In [ ]:
class STAR_KV_Compressor:
    @staticmethod
    def compress(k, v, energy_k=0.95, energy_v=0.90):
        B, H, S, D = k.shape
        _, Sk, _ = torch.linalg.svd(k.reshape(-1, S, D), full_matrices=False)
        rank_k = max(1, min(int(((Sk**2).cumsum(-1)/(Sk**2).sum(-1,keepdim=True).clamp(1e-10) < energy_k).sum(-1).max()+1), S, D))
        _, Sv, _ = torch.linalg.svd(v.reshape(-1, S, D), full_matrices=False)
        rank_v = max(1, min(int(((Sv**2).cumsum(-1)/(Sv**2).sum(-1,keepdim=True).clamp(1e-10) < energy_v).sum(-1).max()+1), S, D))
        k2,v2 = k.reshape(-1,S,D), v.reshape(-1,S,D)
        Uk,Sk,Vhk = RandomSVD.compute(k2, rank_k)
        Uv,Sv,Vhv = RandomSVD.compute(v2, rank_v)
        ko = (Uk*Sk.unsqueeze(-2)@Vhk).reshape(B,H,S,D)
        vo = (Uv*Sv.unsqueeze(-2)@Vhv).reshape(B,H,S,D)
        cb = (Uk.numel()+Sk.numel()+Vhk.numel()+Uv.numel()+Sv.numel()+Vhv.numel())*2
        return ko, vo, rank_k, rank_v, (k.numel()+v.numel())*2/cb if cb>0 else 1.0


In [ ]:
def compress_ultra(k, v, rank=2):
    B,H,S,D = k.shape; r = min(rank,S,D)
    k2,v2 = k.reshape(-1,S,D), v.reshape(-1,S,D)
    Uk,Sk,Vhk = RandomSVD.compute(k2, r)
    Uv,Sv,Vhv = RandomSVD.compute(v2, r)
    Klr = (Uk*Sk.unsqueeze(-2)@Vhk).reshape(B,H,S,D)
    Vlr = (Uv*Sv.unsqueeze(-2)@Vhv).reshape(B,H,S,D)
    cb = (Uk.numel()+Sk.numel()+Vhk.numel()+Uv.numel()+Sv.numel()+Vhv.numel())*2
    return Klr, Vlr, (k.numel()+v.numel())*2/cb if cb>0 else 1.0, r

def lora_restore(Klr, Vlr, lA, lB, s=1.0):
    # LoRA A shape: (in_dim, r) = (768, 8) for c_attn
    # Use a slice: first D rows of A for KV restoration
    B,H,S,D = Klr.shape
    kf = Klr.reshape(-1, D)
    vf = Vlr.reshape(-1, D)
    # Use first D rows of A (maps to K projection subspace)
    A_sub = lA[:D, :]  # (D, r)
    B_sub = lB[:, :D]  # (r, D) if lB shape is (r, out_dim)
    # Actually lB is (r, out_dim) where out_dim=2304 for c_attn
    # Take first D columns of B
    if lB.dim() == 2 and lB.shape[1] >= D:
        B_sub = lB[:, :D]  # (r, D)
    else:
        B_sub = lB.T[:D, :] if lB.shape[0] == D else lB
    try:
        correction = (kf @ A_sub @ B_sub) * s
        return (kf + correction).reshape(B,H,S,D), Vlr
    except Exception as e:
        print(f'LoRA restore error: {e}', flush=True)
        return Klr, Vlr

def train_head(Klr, Kf, lA, r=8, D=64, st=200):
    # Train restoration head: (D) -> (D) mapping, NO LoRA A dependency
    # Use small 2-layer: D -> r -> D
    W1 = nn.Linear(D, r, bias=False)
    W2 = nn.Linear(r, D, bias=False)
    o = torch.optim.Adam(list(W1.parameters())+list(W2.parameters()), lr=1e-2)
    kf,tg = Klr.reshape(-1,D).detach(), Kf.reshape(-1,D).detach()
    for _ in range(st):
        pred = W2(W1(kf))
        l = F.mse_loss(pred, tg); o.zero_grad(); l.backward(); o.step()
    return W1, W2


In [ ]:
class LoRAConv1D(nn.Module):
    def __init__(self, orig, r=8, alpha=16.0):
        super().__init__()
        self.orig = orig
        self.scaling = alpha/r
        self.lora_A = nn.Parameter(torch.randn(orig.weight.shape[0], r)*0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, orig.nf))
        self.active = True
    def activate(self, a=True): self.active = a
    def forward(self, x):
        h = self.orig(x)
        if self.active: h = h+(x@self.lora_A@self.lora_B)*self.scaling
        return h


In [ ]:
class KvForgeModel:
    def __init__(self, model_name='gpt2', lora_rank=8):
        self.device = device
        print(f'Loading {model_name}...', flush=True)
        self.base = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.tok.pad_token = self.tok.eos_token
        cfg = self.base.config
        self.nl = cfg.n_layer if hasattr(cfg,'n_layer') else cfg.num_hidden_layers
        self.hd = (cfg.n_embd // cfg.n_head) if hasattr(cfg,'n_embd') else 64
        self.nh = cfg.n_head if hasattr(cfg,'n_head') else 12
        c = 0
        for n, m in self.base.named_modules():
            if n.endswith('.attn.c_attn') or n.endswith('.attn.c_proj'):
                p, ch = self.base, n.split('.')[-1]
                for pt in n.split('.')[:-1]:
                    if pt: p = getattr(p, pt)
                setattr(p, ch, LoRAConv1D(m, r=lora_rank, alpha=16.0))
                c += 1
        print(f'LoRA: {c} modules', flush=True)
        lp = sum(p.numel() for n,p in self.base.named_parameters() if 'lora' in n)
        tp = sum(p.numel() for p in self.base.parameters())
        print(f'Model: {tp/1e6:.1f}M, LoRA: {lp/1e3:.1f}K, Layers: {self.nl}, Head: {self.hd}', flush=True)

    def set_lora(self, a=True):
        for m in self.base.modules():
            if hasattr(m, 'activate'): m.activate(a)

    def get_kv(self, text):
        inp = self.tok(text, return_tensors='pt', truncation=True, max_length=128).to(device)
        self.set_lora(False)
        out = self.base.generate(input_ids=inp['input_ids'], max_new_tokens=1, use_cache=True,
                                 pad_token_id=self.tok.eos_token_id, do_sample=False, return_dict_in_generate=True)
        past = out.past_key_values
        self.set_lora(False)
        print(f'KV type: {type(past).__name__}', flush=True)
        # Try direct attribute access
        for attr in ['key_cache', '_key_cache', 'cached_keys']:
            if hasattr(past, attr):
                try:
                    obj = getattr(past, attr)
                    if isinstance(obj, (list, tuple)) and len(obj) > 0:
                        k = obj[0]
                        v_attr = attr.replace('key','value').replace('_key','_value')
                        v = getattr(past, v_attr, obj[0])
                        if isinstance(v, (list,tuple)): v = v[0]
                        if isinstance(k, (tuple,list)): k = k[0]
                        if isinstance(v, (tuple,list)): v = v[0]
                        print(f'Extracted via {attr}', flush=True)
                        return k, v, inp['input_ids'].shape[1]
                except:
                    continue
        # Try to_tuple
        if hasattr(past, 'to_tuple'):
            t = past.to_tuple()
            if isinstance(t[0], (tuple,list)):
                k, v = t[0][0], t[0][1]
                print('Extracted via to_tuple', flush=True)
                return k, v, inp['input_ids'].shape[1]
        # Try iteration
        try:
            items = list(past)
            if items and isinstance(items[0], (tuple,list)):
                k, v = items[0][0], items[0][1]
            else:
                k, v = items[0], items[1]
            print('Extracted via list(past)', flush=True)
            return k, v, inp['input_ids'].shape[1]
        except Exception as e:
            attrs = [a for a in dir(past) if not a.startswith('_')][:15]
            raise RuntimeError(f'Cannot extract KV from {type(past)}. attrs: {attrs}. err: {e}')

    def analyse(self, text):
        raw_k, raw_v, slen = self.get_kv(text)
        lora_mod = next((m for n,m in self.base.named_modules() if isinstance(m, LoRAConv1D) and 'c_attn' in n), None)
        B,H,S,D = raw_k.shape
        res = {'seq': slen, 'hd': D, 'nh': H}
        k_skv, v_skv, rk, rv, cr = STAR_KV_Compressor.compress(raw_k, raw_v, energy_k=0.90, energy_v=0.85)
        res['skv'] = {'rk': rk, 'rv': rv, 'cr': cr, 'mse': F.mse_loss(k_skv, raw_k).item()}
        k_lr, v_lr, cr2, ru = compress_ultra(raw_k, raw_v, rank=2)
        mu = F.mse_loss(k_lr, raw_k).item()
        res['ultra'] = {'rk': ru, 'cr': cr2, 'mse': mu}
        if lora_mod:
            kr,_ = lora_restore(k_lr, v_lr, lora_mod.lora_A, lora_mod.lora_B, lora_mod.scaling)
            mk = F.mse_loss(kr, raw_k).item()
            res['lora_raw'] = {'mse': mk, 'imp': max(0, (1-mk/mu)*100)}
        if lora_mod:
            W1, W2 = train_head(k_lr, raw_k, lora_mod.lora_A, r=lora_mod.lora_A.shape[1], D=D, st=200)
            with torch.no_grad():
                p = W2(W1(k_lr.reshape(-1,D))).reshape(raw_k.shape)
                mt = F.mse_loss(p, raw_k).item()
                res['lora_tr'] = {'mse': mt, 'params': (W1.weight.numel()+W2.weight.numel()), 'imp': max(0,(1-mt/mu)*100)}
        return res


In [ ]:
print('='*50)
print('KvForge + STAR-KV + LoRA-RKR — Kaggle')
print('='*50)
m = KvForgeModel('gpt2', lora_rank=8)
r = m.analyse('The transformer architecture revolutionized NLP by introducing self-attention mechanisms that process entire sequences in parallel.')

print(f'\nSeq: {r["seq"]} | Head: {r["hd"]} | Heads: {r["nh"]}')
print(f'{"Method":<22} {"CR":>6} {"MSE(k)":>11}')
print('-'*42)
print(f'{"Full":<22} {1.0:>6.1f}x {0:>11.6f}')
s = r['skv']
print(f'{"STAR-KV (e=0.90)":<22} {s["cr"]:>6.1f}x {s["mse"]:>11.6f}')
u = r['ultra']
print(f'{"Ultra LR (r=2)":<22} {u["cr"]:>6.1f}x {u["mse"]:>11.6f}')
if 'lora_raw' in r:
    print(f'{"+LoRA raw restore":<22} {"—":>6} {r["lora_raw"]["mse"]:>11.6f} ({r["lora_raw"]["imp"]:.0f}%)')
if 'lora_tr' in r:
    t = r['lora_tr']
    print(f'{"+LoRA trained head":<22} {"—":>6} {t["mse"]:>11.6f} ({t["imp"]:.0f}%)')
    print(f'{"  ~params":<22} {t["params"]:>6}')

best_mse = min([x['mse'] for k,x in r.items() if isinstance(x,dict) and 'mse' in x])
best_name = [k for k,x in r.items() if isinstance(x,dict) and 'mse' in x and x['mse']==best_mse][0]
print(f'\nBest: {best_name} (MSE={best_mse:.6f})')

if 'lora_tr' in r and r['lora_tr']['imp'] > 10:
    print('✅ LoRA-RKR trained restoration WORKS!')
elif 'lora_tr' in r:
    print('⚠️ LoRA-RKR: trained head mejba zayif. Real trained LoRA gerekli.')
print('\n✅ Done!')
